# Measure the thickness of a strut from the raw CT TIFF

This notebook measures the diameter of a strut by sampling several cross-sections perpendicular to a centerline supplied in TIFF voxel coordinates. No JSON or other geometry file is used. TIFF arrays are indexed as `[z, y, x]`, while the input center and direction below use `(x, y, z)`.

In [11]:
from pathlib import Path
import sys
import numpy as np

ROOT = Path("/Users/amannindra/Projects/llnl_data_science_challenge_2026")
SCRIPTS = ROOT / "Aman_Scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

from Components.asset_io import read_tiff
from Components.strut_metrology import measure_strut_cross_sections

TIFF_PATH = ROOT / "data/missing_struts/tif_stacks/210127_Brian_Tran_strut_lattices_0point5dash1 1 Slices.tif"
# TIFF_PATH = ROOT / "data/missing_struts/tif_stacks/Tilted_Brian_Tran_segmented.tiff"
VOXEL_PITCH_MM = 0.0581
NOMINAL_DIAMETER_MM = 0.424
EXPECTED_RADIUS_VOXELS = NOMINAL_DIAMETER_MM / (2.0 * VOXEL_PITCH_MM)

# TIFF is memory-mapped by default, so this does not copy the complete volume.
volume_zyx = read_tiff(TIFF_PATH)
print(f"TIFF shape (z, y, x): {volume_zyx.shape}")
print(f"Expected radius: {EXPECTED_RADIUS_VOXELS:.3f} voxels")

TIFF shape (z, y, x): (761, 815, 837)
Expected radius: 3.649 voxels


In [12]:
def measure_strut_thickness(center_xyz, direction_xyz, length_voxels, *, expected_radius_voxels=EXPECTED_RADIUS_VOXELS):
    """Measure one TIFF-only strut from center, direction, and length."""
    center = np.asarray(center_xyz, dtype=float)
    direction = np.asarray(direction_xyz, dtype=float)
    direction /= np.linalg.norm(direction)
    half_length = float(length_voxels) / 2.0
    endpoint0 = center - half_length * direction
    endpoint1 = center + half_length * direction

    measurement = measure_strut_cross_sections(
        volume_zyx,
        endpoint0,
        endpoint1,
        expected_radius_voxels,
        minimum_local_contrast=1000.0,
    )
    radii = np.asarray(
        [value for value in measurement["station_observed_radius_voxels"] if value is not None],
        dtype=float,
    )
    if radii.size == 0:
        raise RuntimeError("No valid cross-section radius was measured for this strut.")

    diameters_voxels = 2.0 * radii
    result = {
        "center_xyz_voxels": center.tolist(),
        "direction_xyz": direction.tolist(),
        "length_voxels": float(length_voxels),
        "measurement_valid": bool(measurement["measurement_valid"]),
        "valid_cross_sections": int(radii.size),
        "diameter_voxels_median": float(np.median(diameters_voxels)),
        "diameter_voxels_p10": float(np.quantile(diameters_voxels, 0.10)),
        "diameter_voxels_p90": float(np.quantile(diameters_voxels, 0.90)),
        "thickness_mm_median": float(np.median(diameters_voxels) * VOXEL_PITCH_MM),
        "thickness_um_median": float(np.median(diameters_voxels) * VOXEL_PITCH_MM * 1000.0),
        "observed_axial_fraction": float(measurement["observed_axial_fraction"]),
        "connected_support": bool(measurement["connected_support"]),
        "endpoint0_xyz_voxels": measurement["endpoint0_xyz_voxels"],
        "endpoint1_xyz_voxels": measurement["endpoint1_xyz_voxels"],
        "raw_measurement": measurement,
    }
    return result

# Replace these three values for any strut visible in the TIFF.
STRUT_CENTER_XYZ = (50.0, 50.0, 50.0)
STRUT_DIRECTION_XYZ = (1.0, 0.0, 0.0)
STRUT_LENGTH_VOXELS = 35.0
thickness = measure_strut_thickness(STRUT_CENTER_XYZ, STRUT_DIRECTION_XYZ, STRUT_LENGTH_VOXELS)
for key in ("center_xyz_voxels", "direction_xyz", "measurement_valid", "valid_cross_sections", "diameter_voxels_median", "thickness_mm_median", "thickness_um_median", "observed_axial_fraction", "connected_support"):
    print(f"{key}: {thickness[key]}")

center_xyz_voxels: [50.0, 50.0, 50.0]
direction_xyz: [1.0, 0.0, 0.0]
measurement_valid: True
valid_cross_sections: 14
diameter_voxels_median: 7.31712971569544
thickness_mm_median: 0.4251252364819051
thickness_um_median: 425.12523648190506
observed_axial_fraction: 0.7777777777777778
connected_support: False


Interpretation: `thickness_mm_median` is the measured diameter, not the radius. The result is a native-TIFF measurement with local contrast and support checks; a low `observed_axial_fraction`, `connected_support=False`, or `measurement_valid=False` means the value should be reviewed rather than treated as ground truth.